# Exercise planning using genetic algorithm

In [102]:
import os
import numpy as np
import pandas as pd
from planner.planner import Planner
from planner.exercise import Exercise
from data.data_loader import DataLoader

1. Load exercises and muscle groups

In [103]:
csv_path = os.path.join("data", "exrx_exercises_muscles_clean.csv")
data_loader = DataLoader(csv_path)

In [104]:
df = data_loader.raw()

In [105]:
df.head()

,exercise_name,exercise_url,body_part,Target,Synergists,Dynamic Stabilizers,Stabilizers,Antagonist Stabilizers
0,Barbell Standing Leg Calf Raise,https://exrx.net/WeightExercises/Gastrocnemius...,Calves,Gastrocnemius,Soleus,NaN,No significant stabilizers,NaN
1,Cable Belt Calf Raise,https://exrx.net/WeightExercises/Gastrocnemius...,Calves,Gastrocnemius,Soleus,NaN,No significant stabilizers,NaN
2,Cable One Arm Single Leg Calf Raise,https://exrx.net/WeightExercises/Gastrocnemius...,Calves,Gastrocnemius,Soleus,NaN,"Trapezius, Upper; Trapezius, Middle; Levator S...",NaN
3,Dumbbell Standing Calf Raise,https://exrx.net/WeightExercises/Gastrocnemius...,Calves,Gastrocnemius,Soleus,NaN,"Trapezius, Upper; Trapezius, Middle; Levator S...",NaN
4,Dumbbell Single Leg Calf Raise,https://exrx.net/WeightExercises/Gastrocnemius...,Calves,Gastrocnemius,Soleus,NaN,"Trapezius, Upper; Trapezius, Middle; Levator S...",NaN


In [106]:
exercieses = data_loader.exercises()

In [107]:
exercieses[:5]

array([Exercise(name='Barbell Standing Leg Calf Raise', targets=['Gastrocnemius'], synergists=['Soleus'], stabilizers=[]),
       Exercise(name='Cable Belt Calf Raise', targets=['Gastrocnemius'], synergists=['Soleus'], stabilizers=[]),
       Exercise(name='Cable One Arm Single Leg Calf Raise', targets=['Gastrocnemius'], synergists=['Soleus'], stabilizers=['Trapezius, Upper', 'Trapezius, Middle', 'Levator Scapulae', 'Gluteus Medius', 'Gluteus Minimus', 'Quadratus        Lumborum', 'Obliques']),
       Exercise(name='Dumbbell Standing Calf Raise', targets=['Gastrocnemius'], synergists=['Soleus'], stabilizers=['Trapezius, Upper', 'Trapezius, Middle', 'Levator Scapulae', 'Gluteus Medius', 'Gluteus Minimus']),
       Exercise(name='Dumbbell Single Leg Calf Raise', targets=['Gastrocnemius'], synergists=['Soleus'], stabilizers=['Trapezius, Upper', 'Trapezius, Middle', 'Levator Scapulae', 'Gluteus Medius', 'Gluteus Minimus', 'Quadratus        Lumborum', 'Obliques'])],
      dtype=object)

2. Define hiperparameters

In [108]:
POPULATION_SIZE = 2
NUM_EXERCISES_TO_PLAN = 8
MUSCLE_GROUP_WEIGHTS = np.ones(data_loader.muscle_group_count())
INTENSITY_WEIGHTS = np.array([1.0, 1.5, 2.0])
BALANCE_WEIGHT = 1.0

3. Create random plan

In [109]:
planner = Planner(
    exercises=exercieses,
    num_muscle_groups=data_loader.muscle_group_count(),
    muscle_group_weights=MUSCLE_GROUP_WEIGHTS,
    intensity_weights=INTENSITY_WEIGHTS,
    balance_weight=BALANCE_WEIGHT
)


In [110]:
population = planner.initialize_population(POPULATION_SIZE, NUM_EXERCISES_TO_PLAN)

In [111]:
population

array([[200, 644, 519, 113,  85, 163, 338,   6],
       [ 99, 568, 168, 141, 277, 590, 381, 609]], dtype=int32)

In [112]:
fitnesses = planner.evaluate_fitness(population)
fitnesses

array([-71.71301765, -85.13029144])

4. Optimize the population with Genetic.next_generation

In [113]:
from algorithms.genetic.genetic import Genetic

def ga_fitness(individual: np.ndarray) -> np.float64:
    return np.float64(planner.fitness_function(individual))

ga = Genetic(
    fitness_function=ga_fitness,
    sequence_values=np.arange(len(exercieses), dtype=np.int32),
    crossover_type="two-point",
    mutation_chance=0.1,
    rng=np.random.default_rng(42),
)

ga_population = population.copy()
ga_fitness_scores = np.array([ga_fitness(ind) for ind in ga_population])
print("Initial GA cost (lower is better):", ga_fitness_scores)

Initial GA cost (lower is better): [-71.71301765 -85.13029144]


In [114]:
NUM_GENERATIONS = 1000
ELITE_COUNT = 1

for generation in range(1, NUM_GENERATIONS + 1):
    ga_population = ga.next_generation(ga_population, elite_count=ELITE_COUNT)
    ga_fitness_scores = np.array([ga_fitness(ind) for ind in ga_population])
    best_idx = int(np.argmin(ga_fitness_scores))
    if generation % 100 == 0:
        print(
            f"Generation {generation:02d} | best_idx={best_idx} | ",
            f"best_cost={ga_fitness_scores[best_idx]:.4f}",
        )

best_idx = int(np.argmin(ga_fitness_scores))
best_individual = ga_population[best_idx]
print("Best individual indices:", best_individual)

Generation 100 | best_idx=1 |  best_cost=-49.0167
Generation 200 | best_idx=0 |  best_cost=-7.8937
Generation 300 | best_idx=0 |  best_cost=-7.2043
Generation 400 | best_idx=0 |  best_cost=-6.9775
Generation 500 | best_idx=1 |  best_cost=-19.4032
Generation 600 | best_idx=1 |  best_cost=-17.0319


Generation 700 | best_idx=0 |  best_cost=-4.3858
Generation 800 | best_idx=0 |  best_cost=-4.3858
Generation 900 | best_idx=0 |  best_cost=-2.6798
Generation 1000 | best_idx=0 |  best_cost=-2.0426
Best individual indices: [ 42 223  42 148 434 175  35 223]


In [115]:
best_cost = planner.fitness_function(best_individual)
best_intensity_matrix = planner.get_intensity_matrix(best_individual)

print("Best plan cost (lower is better):", float(best_cost))
print("Best plan intensity matrix:")
print(best_intensity_matrix)

# Optional: show selected exercises
best_exercises = [exercieses[idx].name for idx in best_individual]
pd.Series(best_exercises, name="exercise_name")

Best plan cost (lower is better): -2.042626521107529
Best plan intensity matrix:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]]


0     Sled 45° Reverse Calf Raise (plate loaded)
1                         Suspended Pull Through
2     Sled 45° Reverse Calf Raise (plate loaded)
3                          Crunch (arms crossed)
4                                   Inverted Row
5                     Cable Isolateral Push Pull
6    Lever 45° Reverse Calf Press (plate loaded)
7                         Suspended Pull Through
Name: exercise_name, dtype: str